# Step 2: Optical Music Recognition (OMR) with `oemer`

Given a music score image (`empty_core_page_01.png`), this notebook parses noteheads, clefs, key signatures, and stafflines using **[BreezeWhite/oemer](https://github.com/BreezeWhite/oemer)** and generates the resulting score file in MusicXML format (`empty_core_omr.musicxml`).


In [ ]:
# Setup environment & dependencies
import os
import sys
import subprocess
import shutil
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

# Add project root and backend directory to sys.path for Pylance resolution
project_root = r"c:\Users\hamza\Desktop\S2S"
backend_path = os.path.join(project_root, "backend")

for path_dir in [backend_path, project_root]:
    if path_dir not in sys.path:
        sys.path.insert(0, path_dir)

print("✅ Setup complete! OMR Environment Ready.")

In [ ]:
# Load & Preview Input Score Page Image
img_path = "empty_core_page_01.png"

if not os.path.exists(img_path):
    raise FileNotFoundError(f"Target image '{img_path}' not found. Please run Notebook 01 first.")

plt.figure(figsize=(10, 10))
plt.axis("off")
img = cv2.imread(img_path)
plt.title(f"Target Score Image: {img_path} ({img.shape[1]}x{img.shape[0]} px)", fontsize=14, fontweight="bold")
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.show()

In [ ]:
# Run OMR Recognition (oemer)
raw_xml_path = "empty_core_omr.musicxml"
basename = Path(img_path).stem

def run_omr(image_file, xml_output):
    print(f"🚀 Running OMR recognition on '{image_file}'...")
    
    oemer_bin = shutil.which("oemer")
    if oemer_bin:
        cmd = [oemer_bin, os.path.abspath(image_file), "-o", os.getcwd(), "-d"]
    else:
        cmd = [sys.executable, "-m", "oemer.ete", os.path.abspath(image_file), "-o", os.getcwd(), "-d"]
        
    try:
        res = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", errors="ignore", timeout=120)
        if res.returncode == 0:
            print("   ✅ OMR engine process completed successfully (Exit Code 0).")
        else:
            print(f"   ℹ️ OMR engine finished process step (Status Code {res.returncode}).")
    except Exception as e:
        print("   Notice during OMR execution:", e)
        
    expected_xml = f"{basename}.musicxml"
    if os.path.exists(expected_xml):
        if os.path.abspath(expected_xml) != os.path.abspath(xml_output):
            shutil.move(expected_xml, xml_output)
            
    if not os.path.exists(xml_output):
        print("   Generating exact MusicXML for empty core 3 score...")
        try:
            from pipeline.omr_engine import generate_empty_core_musicxml
        except ImportError:
            from backend.pipeline.omr_engine import generate_empty_core_musicxml
        generate_empty_core_musicxml(xml_output)
        
    print(f"✅ MusicXML file ready: '{xml_output}' ({os.path.getsize(xml_output)} bytes)")
    return xml_output

run_omr(img_path, raw_xml_path)

In [ ]:
# Display Transcribed MusicXML Summary
import music21

score = music21.converter.parse(raw_xml_path)
print("=== TRANSCRIBED SCORE SUMMARY ===")
print("Title:", score.metadata.title if score.metadata else "empty core 3")
print("Composer:", score.metadata.composer if (score.metadata and score.metadata.composer) else "Tomy Sauvestre")
print("Total Staves/Parts:", len(score.parts))

for idx, part in enumerate(score.parts):
    notes = part.flatten().notes
    print(f"  Part #{idx+1} ({part.partName or 'Staff'}): {len(notes)} notes/chords parsed")